# MVP do Sprint Análise de Dados e Boas Práticas

- **Aluno:** Diogo Reis Beltrão Lessa
- **Matrícula:** 4052025002548  
- **Curso:** Pós-Graduação em Ciência de Dados e Analytics - PUC-Rio

---

## Dataset

**Credit Card Customers (BankChurners)**  
Fonte: [Kaggle — sakshigoyal7/credit-card-customers](https://www.kaggle.com/datasets/sakshigoyal7/credit-card-customers)

| Característica | Valor |
|---|---|
| Registros | 10.127 clientes |
| Variáveis originais | 23 (21 utilizáveis) |
| Variável-alvo | `Attrition_Flag` (Existing Customer / Attrited Customer) |


---

## Objetivo

Analisar o perfil e o comportamento de clientes de cartão de crédito para identificar os fatores mais associados ao **churn** (cancelamento do cartão), aplicando técnicas de análise exploratória de dados (EDA) e pré-processamento.

# Descrição do Problema

O dataset **Credit Card Customers (BankChurners)**, disponível no Kaggle, simula o cenário de um banco que observa uma taxa de cancelamento de aproximadamente **16%** nos cartões de crédito. Segundo a descrição original do dataset, o contexto é o de um gerente bancário que precisa prever quais clientes vão cancelar seus cartões para poder atuar preventivamente na retenção.

A taxa de ~16% é derivada da própria distribuição da variável-alvo no dataset: aproximadamente **1.627 clientes classificados como "Attrited Customer"** em um total de **10.127 registros**. Os demais ~8.500 clientes permanecem ativos ("Existing Customer").

O objetivo deste trabalho é identificar, por meio de **análise exploratória de dados (EDA) e pré-processamento**, quais características dos clientes estão mais associadas ao churn — preparando os dados para um futuro modelo preditivo de retenção. A etapa de modelagem não faz parte do escopo deste MVP.

## Hipóteses do Problema

Com base no domínio de negócio e nas variáveis disponíveis, levantamos as seguintes hipóteses a serem investigadas e respondidas ao longo deste notebook:

**H1 — Inatividade e churn:**  
Clientes com mais meses de inatividade nos últimos 12 meses (`Months_Inactive_12_mon`) apresentam maior taxa de cancelamento do cartão.

**H2 — Utilização do crédito e churn:**  
Clientes com baixa taxa de utilização do limite de crédito (`Avg_Utilization_Ratio`) têm maior propensão ao churn, indicando desengajamento com o produto.

**H3 — Número de produtos bancários e churn:**  
Clientes com menor número de produtos contratados com o banco (`Total_Relationship_Count`) cancelam o cartão com maior frequência, sugerindo menor vínculo com a instituição.

**H4 — Faixa de renda e churn:**  
A faixa de renda do cliente (`Income_Category`) influencia a probabilidade de cancelamento — clientes de renda mais baixa podem ter maior pressão financeira para cancelar, enquanto clientes de renda alta podem buscar produtos mais vantajosos em outros bancos.

**H5 — Redução de transações e churn:**  
Clientes que reduziram sua contagem de transações do Q1 para o Q4 (`Total_Ct_Chng_Q4_Q1` com valores menores indicando queda) apresentam maior risco de cancelamento, revelando uma mudança comportamental que precede o churn.

## Tipo de Problema

Trata-se de um problema de **classificação supervisionada binária**:

- **Variável-alvo:** `Attrition_Flag`
  - `"Existing Customer"` → cliente ativo (classe majoritária, ~84%)
  - `"Attrited Customer"` → cliente que cancelou (classe minoritária, ~16%)

O **escopo deste MVP** abrange exclusivamente as etapas de **Análise Exploratória de Dados (EDA)** e **Pré-Processamento**. A etapa de modelagem preditiva (treinamento e avaliação de classificadores) é um próximo passo natural, mas está fora do escopo atual.

O desbalanceamento de classes (~84%/16%) é uma característica relevante que será discutida ao longo do notebook e deverá ser tratada na etapa de modelagem futura com técnicas como SMOTE ou ajuste de pesos de classe.

## Seleção de Dados

**Fonte:** Kaggle — [Credit Card Customers](https://www.kaggle.com/datasets/sakshigoyal7/credit-card-customers), publicado por sakshigoyal7.

**Dimensões:** 10.127 instâncias × 23 colunas originais (21 utilizáveis após remoção de `CLIENTNUM` e das 2 colunas de saída do Naive Bayes inseridas pelo autor do dataset).

**Justificativa para a escolha deste dataset:**

- **Riqueza de variáveis:** mistura equilibrada de variáveis numéricas (14) e categóricas (5), exigindo diferentes estratégias de encoding e pré-processamento.
- **Missing values disfarçados:** a categoria `"Unknown"` presente em três colunas (`Education_Level`, `Marital_Status`, `Income_Category`) representa um caso realista de dado faltante implícito — comum em datasets de negócio.
- **Desbalanceamento de classes:** a proporção ~84%/16% é típica de problemas de churn e exige atenção especial, sendo um excelente exercício de análise crítica.
- **Contexto de negócio claro:** o problema de retenção de clientes é relevante e bem delimitado, facilitando a formulação de hipóteses e a interpretação dos resultados.
- **Potencial para múltiplas técnicas:** o dataset é adequado para demonstrar normalização, padronização, encoding ordinal, One-Hot Enconding (OHE), detecção de outliers, transformação logarítmica e feature engineering.

## Atributos do Dataset

O dataset original possui 23 colunas. Após remoção de `CLIENTNUM` (identificador sem valor preditivo) e das 2 últimas colunas (outputs do modelo Naive Bayes inseridos pelo autor), restam **21 variáveis utilizáveis**:

| # | Variável | Tipo | Descrição |
|---|---|---|---|
| 1 | `Attrition_Flag` | Categórica (target) | Status do cliente: `"Existing Customer"` ou `"Attrited Customer"` |
| 2 | `Customer_Age` | Numérica (int) | Idade do cliente em anos |
| 3 | `Gender` | Categórica binária | Gênero: `M` ou `F` |
| 4 | `Dependent_count` | Numérica (int) | Número de dependentes (0–5) |
| 5 | `Education_Level` | Categórica ordinal | Escolaridade: Uneducated, High School, College, Graduate, Post-Graduate, Doctorate, **Unknown** |
| 6 | `Marital_Status` | Categórica nominal | Estado civil: Married, Single, Divorced, **Unknown** |
| 7 | `Income_Category` | Categórica ordinal | Faixa de renda: Less than $40K, $40K - $60K, $60K - $80K, $80K - $120K, $120K +, **Unknown** |
| 8 | `Card_Category` | Categórica ordinal | Tipo de cartão: Blue, Silver, Gold, Platinum |
| 9 | `Months_on_book` | Numérica (int) | Tempo como cliente do banco (em meses) |
| 10 | `Total_Relationship_Count` | Numérica (int) | Total de produtos contratados com o banco (1–6) |
| 11 | `Months_Inactive_12_mon` | Numérica (int) | Meses inativos nos últimos 12 meses |
| 12 | `Contacts_Count_12_mon` | Numérica (int) | Número de contatos com o banco nos últimos 12 meses |
| 13 | `Credit_Limit` | Numérica (float) | Limite de crédito do cartão |
| 14 | `Total_Revolving_Bal` | Numérica (int) | Saldo rotativo total (quanto o cliente deve no cartão) |
| 15 | `Avg_Open_To_Buy` | Numérica (float) | Limite disponível médio (Credit_Limit − Total_Revolving_Bal) |
| 16 | `Total_Amt_Chng_Q4_Q1` | Numérica (float) | Variação do valor transacionado entre Q4 e Q1 |
| 17 | `Total_Trans_Amt` | Numérica (int) | Valor total transacionado no período |
| 18 | `Total_Trans_Ct` | Numérica (int) | Contagem total de transações no período |
| 19 | `Total_Ct_Chng_Q4_Q1` | Numérica (float) | Variação da contagem de transações entre Q4 e Q1 |
| 20 | `Avg_Utilization_Ratio` | Numérica (float) | Taxa média de utilização do crédito (0.0–1.0) |



> **Nota sobre "Unknown":** As colunas `Education_Level`, `Marital_Status` e `Income_Category` contêm a categoria `"Unknown"` como proxy de valores faltantes. As quantidades exatas serão calculadas a partir do CSV na etapa de EDA (Bloco 4). Serão avaliadas três estratégias de tratamento no pré-processamento (Bloco 5).

# Importação das Bibliotecas Necessárias e Carga de Dados

Esta seção centraliza todos os imports do projeto e o carregamento do dataset. Manter os imports em um único bloco é uma boa prática que facilita a leitura, evita dependências ocultas entre células e simplifica a adaptação do notebook para outros ambientes.

As bibliotecas utilizadas cobrem:
- **Manipulação de dados:** `pandas`, `numpy`
- **Visualização:** `matplotlib`, `seaborn`
- **Pré-processamento:** `scikit-learn` (scalers, encoders, split)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Configurações globais de visualização
sns.set_theme(style='whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("Bibliotecas carregadas com sucesso.")

Bibliotecas carregadas com sucesso.


In [2]:
# Carregamento do dataset a partir do repositório GitHub
url = 'https://raw.githubusercontent.com/diogorblessa/mvp-credit-card-churn/main/data/BankChurners.csv'
df = pd.read_csv(url)

print(f"Dataset carregado: {df.shape[0]} linhas x {df.shape[1]} colunas")
df.head()

Dataset carregado: 10127 linhas x 23 colunas


,CLIENTNUM,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,...,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2
0,768805383,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,...,12691.0,777,11914.0,1.335,1144,42,1.625,0.061,0.000093,0.99991
1,818770008,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,...,8256.0,864,7392.0,1.541,1291,33,3.714,0.105,0.000057,0.99994
2,713982108,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,...,3418.0,0,3418.0,2.594,1887,20,2.333,0.000,0.000021,0.99998
3,769911858,Existing Customer,40,F,4,High School,Unknown,Less than $40K,Blue,34,...,3313.0,2517,796.0,1.405,1171,20,2.333,0.760,0.000134,0.99987
4,709106358,Existing Customer,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,...,4716.0,0,4716.0,2.175,816,28,2.500,0.000,0.000022,0.99998


## Limpeza Inicial

Antes de iniciar qualquer análise, removemos as colunas que não devem fazer parte do dataset utilizável:

1. **`CLIENTNUM`** — identificador único do cliente. Não possui valor preditivo e sua presença poderia causar vazamento de dados (*data leakage*) em um modelo futuro.

2. **Colunas `Naive_Bayes_Classifier_*` (as 2 últimas)** — são outputs de um modelo Naive Bayes que o autor do dataset original inseriu no arquivo. Não são variáveis descritivas dos clientes; incluí-las seria uma forma grave de *data leakage*, pois já contêm a resposta sobre churn.

Após a remoção, verificamos a presença de **linhas duplicadas**, que poderiam distorcer estatísticas e pesos em modelagem futura.

In [3]:
# Remover CLIENTNUM (identificador sem valor preditivo)
df = df.drop(columns=['CLIENTNUM'])

# Remover as 2 últimas colunas (outputs do Naive Bayes inseridos pelo autor do dataset)
colunas_naive_bayes = [col for col in df.columns if col.startswith('Naive_Bayes_Classifier_')]
print(f"Colunas Naive Bayes removidas: {colunas_naive_bayes}")
df = df.drop(columns=colunas_naive_bayes)

# Verificar duplicatas
n_duplicatas = df.duplicated().sum()
print(f"\nLinhas duplicadas: {n_duplicatas}")

# Shape final
print(f"\nShape após limpeza: {df.shape[0]} linhas × {df.shape[1]} colunas")
print(f"Colunas restantes: {list(df.columns)}")
df.head()

Colunas Naive Bayes removidas: ['Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1', 'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2']

Linhas duplicadas: 0

Shape após limpeza: 10127 linhas × 20 colunas
Colunas restantes: ['Attrition_Flag', 'Customer_Age', 'Gender', 'Dependent_count', 'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category', 'Months_on_book', 'Total_Relationship_Count', 'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 'Credit_Limit', 'Total_Revolving_Bal', 'Avg_Open_To_Buy', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt', 'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio']


,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio
0,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,5,1,3,12691.0,777,11914.0,1.335,1144,42,1.625,0.061
1,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,6,1,2,8256.0,864,7392.0,1.541,1291,33,3.714,0.105
2,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,4,1,0,3418.0,0,3418.0,2.594,1887,20,2.333,0.000
3,Existing Customer,40,F,4,High School,Unknown,Less than $40K,Blue,34,3,4,1,3313.0,2517,796.0,1.405,1171,20,2.333,0.760
4,Existing Customer,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,5,1,0,4716.0,0,4716.0,2.175,816,28,2.500,0.000
